In [2]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

▸,:,


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [ ]:
#Cohort_Lab
Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_lab_Paired_additionalCols_Feb1924.parquet")

In [ ]:
# Dropping a column ("loincclass" in this case)
Cohort_Lab = Cohort_Lab.drop("loincclass")
Cohort_Lab.printSchema()

In [ ]:
#Read Modified Cohort Lab
result_Cohort_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Modified_Lab")

In [ ]:
from pyspark.sql.functions import to_date

# Assuming df is the DataFrame where you want to convert the servicedate column
result_Cohort_df = result_Cohort_df.withColumn("servicedate", to_date(result_Cohort_df["servicedate"], 'yyyy-MM-dd'))

In [ ]:
# List of columns to drop
columns_to_drop = ["modifier", "refLowtype", "refHightype", "refLowtextvalue", "refHightextvalue"]

# Dropping multiple columns
result_Cohort_df = result_Cohort_df.drop(*columns_to_drop)
# Print the schema of the DataFrame to verify column names
result_Cohort_df.printSchema()

# Check if 'servicedate' column is present in the DataFrame
if 'servicedate' in result_Cohort_df.columns:
    print("'servicedate' column exists in the DataFrame")
else:
    print("'servicedate' column does not exist in the DataFrame")

In [ ]:
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define accepted values
accepted_values = ['Abnormal', 'Low', 'High', 'Normal', 'Positive', 'Negative']

# Define a window specification to partition by labcode and order by servicedate
labcode_window = Window.partitionBy("labcode").orderBy(F.col("servicedate").desc())

# Count the frequency of each distinct combination of updated_Interpretation
# (Except for Null/Unknown/Not applicable/Not established) and labcode values
# and include corresponding servicedate
interpretation_counts = result_Cohort_df.groupBy(
    "labcode", "updated_Interpretation", "personid", "servicedate"
).agg(
    F.count("*").alias("count")
)

# Filter the records where updated_Interpretation is either null or not in accepted_values
filtered_df = result_Cohort_df.filter(
    (col("updated_Interpretation").isNull()) |
    (~col("updated_Interpretation").isin(accepted_values))
)

# Get the most frequent updated_Interpretation value for each labcode and personid
max_count_per_labcode = interpretation_counts.withColumn(
    "max_count", F.max("count").over(Window.partitionBy("labcode"))
).filter(
    (F.col("count") == F.col("max_count")) &
    (F.col("updated_Interpretation").isin(accepted_values))
).select(
    F.col("personid").alias("max_personid"),
    F.col("labcode").alias("max_count_labcode"),
    F.col("updated_Interpretation").alias("max_count_updated_Interpretation"),
    F.col("max_count"),
    F.col("servicedate").alias("mx_servicedate")
)

# Get the latest service date within each group
window_spec = Window.partitionBy("max_personid", "max_count_labcode", "max_count").orderBy(col("mx_servicedate").desc())
ranked_df = max_count_per_labcode.withColumn("rank", F.row_number().over(window_spec))
latest_records_df = ranked_df.filter(ranked_df["rank"] == 1).drop("rank")

# Get the count of distinct interpretations within each group
grouped_df = max_count_per_labcode.groupBy("max_personid", "max_count_labcode", "max_count")
count_distinct_interp = grouped_df.agg(F.countDistinct("max_count_updated_Interpretation").alias("distinct_count"))

# Filter out groups where there are multiple distinct interpretations
filtered_groups = count_distinct_interp.filter(count_distinct_interp["distinct_count"] > 1)

# Apply the tie-breaking logic for groups with multiple interpretations
tie_breaker_window = Window.partitionBy("max_personid", "max_count_labcode", "max_count") \
    .orderBy(F.when(latest_records_df["max_count_updated_Interpretation"].isin(accepted_values), 0).otherwise(1),
             F.desc("mx_servicedate"))

tie_breaker_df = latest_records_df.withColumn("row_number", F.row_number().over(tie_breaker_window)) \
    .filter("row_number = 1").drop("row_number")

# Rename columns in filtered_groups to avoid conflicts after the join
filtered_groups_renamed = filtered_groups.withColumnRenamed("max_count", "filtered_max_count")

# Perform an inner join between tie_breaker_df and filtered_groups_renamed
joined_df = tie_breaker_df.join(filtered_groups_renamed, 
                                (tie_breaker_df["max_personid"] == filtered_groups_renamed["max_personid"]) &
                                (tie_breaker_df["max_count_labcode"] == filtered_groups_renamed["max_count_labcode"]) &
                                (tie_breaker_df["max_count"] == filtered_groups_renamed["filtered_max_count"]),
                                "inner") \
                          .select(tie_breaker_df["max_personid"], 
                                  tie_breaker_df["max_count_labcode"], 
                                  tie_breaker_df["max_count"], 
                                  tie_breaker_df["max_count_updated_Interpretation"], 
                                  tie_breaker_df["mx_servicedate"], 
                                  filtered_groups_renamed["distinct_count"])

final_result_df = joined_df

final_result_df.show(100,truncate=False)

In [ ]:
#Control_Lab
Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Lab_Paired_Final_additionalColsFV.parquet")

In [ ]:
#Step1 : Filter Records having Ref Ranges, Value, Interpretation as Null or Unknown or Not applicable and create updated_Interpretation new column and update values as NULL
from pyspark.sql.functions import when, col
# Add a new column with updated interpretation
updated_df1 = Control_Lab.withColumn(
    "updated_Interpretation",
    when((col("interpretation").isNull() | (col("interpretation") == "Unknown") | (col("interpretation") == "Not applicable")) &
         (col("value").isNull() | (col("value") == "Unknown") | (col("value") == "Not applicable")) &
         (col("refLowRange").isNull() | (col("refLowRange") == "Unknown") | (col("refLowRange") == "Not applicable")) &
         (col("refHighRange").isNull() | (col("refHighRange") == "Unknown") | (col("refHighRange") == "Not applicable")),
         None
    ).otherwise(col("interpretation"))
)

# Show the updated DataFrame
updated_df1.show(truncate=False)

In [ ]:
#Step2: Update Records based on condition , Low, Normal, High and retain original value in updated_Interpretation otherwise

from pyspark.sql.functions import when, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
# Print the schema to understand the data types and structure
print("Schema before transformations:")
updated_df1.printSchema()

# Cast columns to double data type for numeric comparison
updated_df1 = updated_df1.withColumn("value", col("value").cast(DoubleType()))
updated_df1 = updated_df1.withColumn("refLowRange", col("refLowRange").cast(DoubleType()))
updated_df1 = updated_df1.withColumn("refHighRange", col("refHighRange").cast(DoubleType()))

# Print the DataFrame to inspect the data after casting
print("DataFrame after casting to double:")
updated_df1.show(truncate=False)

# Define conditions to check if value, refLowRange, and refHighRange columns have numeric values
numeric_conditions = (
    col("value").isNotNull() &
    col("refLowRange").isNotNull() &
    col("refHighRange").isNotNull()
)

# Print the DataFrame with the added column for numeric_conditions
print("DataFrame with numeric_conditions column:")
updated_df1.withColumn("numeric_conditions", numeric_conditions).show(truncate=False)

# Update updated_Interpretation column based on conditions
result_Control_df = updated_df1.withColumn(
    "updated_Interpretation",
    when(
        (numeric_conditions) & (col("value") <= col("refLowRange")), "Low"
    )
    .when(
        (numeric_conditions) & (col("value") >= col("refHighRange")), "High"
    )
    .when(
        (numeric_conditions) & (col("value") > col("refLowRange")) & (col("value") < col("refHighRange")), "Normal"
    )
    .otherwise(col("updated_Interpretation"))  # If value is within the reference range, keep the original interpretation
)

# Show the updated DataFrame
print("Final DataFrame:")
result_Control_df.show(truncate=False)

In [ ]:
#Step3: Update updated_Interpretation column for Records having value or refLowRange or refHighRange or all are null or Not applicable or Unknown or empty with interpretation column values

from pyspark.sql.functions import when, col, lit

# Update 'updated_Interpretation' column based on conditions
result_Control_df = result_Control_df.withColumn(
    "updated_Interpretation",
    when(
        (col("value").isNull()) | 
        (col("refLowRange").isNull()) | 
        (col("refHighRange").isNull()) | 
        (col("value").isin("", "Not applicable", "Unknown")) | 
        (col("refLowRange").isin("", "Not applicable", "Unknown")) | 
        (col("refHighRange").isin("", "Not applicable", "Unknown")), 
        col("interpretation")
    )
    .otherwise(col("updated_Interpretation"))  # Keep the original value if conditions are not met
)

# Show the updated DataFrame
print("Final DataFrame:")
result_Control_df.show(truncate=False)

In [ ]:
#Step 4: Categorizing the updated_Interpretation-> "Below lower panic limits, Below low normal, Below absolute low-off instrument scale as Low",
#"and Above absolute high-off instrument scale, Above high normal, Above upper panic limits as High and Very abnormal as Abnormal"
from pyspark.sql.functions import when, col

# Update 'updated_Interpretation' column based on specific values
result_Control_df = result_Control_df.withColumn(
    "updated_Interpretation",
    when(
        col("updated_Interpretation").like("%Below lower panic limits%") |
        col("updated_Interpretation").like("%Below low normal%") |
        col("updated_Interpretation").like("%Below absolute low-off instrument scale%"),
        "Low"
    )
    .when(
        col("updated_Interpretation").like("%Above absolute high-off instrument scale%") |
        col("updated_Interpretation").like("%Above high normal%") |
        col("updated_Interpretation").like("%Above upper panic limits%"),
        "High"
    )
    .when(
        col("updated_Interpretation").like("%Very abnormal%"),
        "Abnormal"
    )
    .otherwise(col("updated_Interpretation"))  # Keep the original value if conditions are not met
)

# Show the updated DataFrame
print("Final DataFrame:")
result_Control_df.show(truncate=False)

In [ ]:
#Step 5:Update 'updated_Interpretation' as 'Not applicable' for the records having 'updated_Interpretation' as Trace, High risk of, Better, Chlamydia trachomatis DNA [Presence] in Specimen by Probe with signal amplification,
# Specimen source identified, Neisseria gonorrhoeae DNA [Presence] in Specimen by NAA with probe detection
from pyspark.sql.functions import when, col

# List of values to be updated to 'Not applicable'
values_to_update = [
    "Trace",
    "High risk of",
    "Better",
    "Chlamydia trachomatis DNA [Presence] in Specimen by Probe with signal amplification",
    "Specimen source identified",
    "Neisseria gonorrhoeae DNA [Presence] in Specimen by NAA with probe detection"
]

# Update 'updated_Interpretation' column for specified values
result_Control_df = result_Control_df.withColumn(
    "updated_Interpretation",
    when(
        col("updated_Interpretation").isin(values_to_update),
        "Not applicable"
    )
    .otherwise(col("updated_Interpretation"))  # Keep the original value if conditions are not met
)

# Show the updated DataFrame
print("Final DataFrame:")
result_Control_df.show(truncate=False)

In [ ]:
# Repartition based on the number of partitions obtained
result_Control_df = result_Control_df.repartition("updated_Interpretation")

# Write to HDFS with Spark selecting the repartition
result_Control_df.write.mode("overwrite").parquet("Control_Modified_Lab")

In [ ]:
!hadoop fs -copyToLocal Control_Modified_Lab ~/work/Oklahoma%20State/Priya/epilepsy

In [ ]:
from pyspark.sql.functions import broadcast

# Get the number of partitions from the RDD
num_partitions = result_Control_df.rdd.getNumPartitions()

# # Broadcast the DataFrame
# result_Control_df = broadcast(result_Control_df)

# Repartition based on the number of partitions obtained
result_Control_df = result_Control_df.repartition(num_partitions)

# Write to HDFS with Spark selecting the repartition
result_Control_df.write.mode("overwrite").parquet("Control_Modified_Lab")

In [3]:
#Read Modified Cohort Lab
result_Control_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Modified_Lab")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("count of result_Control_df", result_Control_df.count())

In [ ]:
print("count of Control_Lab", Control_Lab.count())

In [ ]:
result_Control_df.printSchema()

In [4]:
from pyspark.sql.functions import to_date

# Assuming df is the DataFrame where you want to convert the servicedate column
result_Control_df = result_Control_df.withColumn("servicedate", to_date(result_Control_df["servicedate"], 'yyyy-MM-dd'))

▸,:,


In [5]:
# List of columns to drop
columns_to_drop = ["modifier", "refLowtype", "refHightype", "refLowtextvalue", "refHightextvalue"]

# Dropping multiple columns
result_Control_df = result_Control_df.drop(*columns_to_drop)
# Print the schema of the DataFrame to verify column names
result_Control_df.printSchema()

# Check if 'servicedate' column is present in the DataFrame
if 'servicedate' in result_Control_df.columns:
    print("'servicedate' column exists in the DataFrame")
else:
    print("'servicedate' column does not exist in the DataFrame")

▸,:,


root
 |-- value: double (nullable = true)
 |-- refLowRange: double (nullable = true)
 |-- refHighRange: double (nullable = true)
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: date (nullable = true)
 |-- updated_Interpretation: string (nullable = true)

'servicedate' column exists in the DataFrame


In [6]:
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define accepted values
accepted_values = ['Abnormal', 'Low', 'High', 'Normal', 'Positive', 'Negative']

# Define a window specification to partition by labcode and order by servicedate
labcode_window = Window.partitionBy("labcode").orderBy(F.col("servicedate").desc())

# Count the frequency of each distinct combination of updated_Interpretation
# (Except for Null/Unknown/Not applicable/Not established) and labcode values
# and include corresponding servicedate
interpretation_counts = result_Control_df.groupBy(
    "labcode", "updated_Interpretation", "personid", "servicedate"
).agg(
    F.count("*").alias("count")
)

# Filter the records where updated_Interpretation is either null or not in accepted_values
filtered_df = result_Control_df.filter(
    (col("updated_Interpretation").isNull()) |
    (~col("updated_Interpretation").isin(accepted_values))
)

# Get the most frequent updated_Interpretation value for each labcode and personid
max_count_per_labcode = interpretation_counts.withColumn(
    "max_count", F.max("count").over(Window.partitionBy("labcode"))
).filter(
    (F.col("count") == F.col("max_count")) &
    (F.col("updated_Interpretation").isin(accepted_values))
).select(
    F.col("personid").alias("max_personid"),
    F.col("labcode").alias("max_count_labcode"),
    F.col("updated_Interpretation").alias("max_count_updated_Interpretation"),
    F.col("max_count"),
    F.col("servicedate").alias("mx_servicedate")
)

# Get the latest service date within each group
window_spec = Window.partitionBy("max_personid", "max_count_labcode", "max_count").orderBy(col("mx_servicedate").desc())
ranked_df = max_count_per_labcode.withColumn("rank", F.row_number().over(window_spec))
latest_records_df = ranked_df.filter(ranked_df["rank"] == 1).drop("rank")

# Get the count of distinct interpretations within each group
grouped_df = max_count_per_labcode.groupBy("max_personid", "max_count_labcode", "max_count")
count_distinct_interp = grouped_df.agg(F.countDistinct("max_count_updated_Interpretation").alias("distinct_count"))

# Filter out groups where there are multiple distinct interpretations
filtered_groups = count_distinct_interp.filter(count_distinct_interp["distinct_count"] > 1)

# Apply the tie-breaking logic for groups with multiple interpretations
tie_breaker_window = Window.partitionBy("max_personid", "max_count_labcode", "max_count") \
    .orderBy(F.when(latest_records_df["max_count_updated_Interpretation"].isin(accepted_values), 0).otherwise(1),
             F.desc("mx_servicedate"))

tie_breaker_df = latest_records_df.withColumn("row_number", F.row_number().over(tie_breaker_window)) \
    .filter("row_number = 1").drop("row_number")

# Rename columns in filtered_groups to avoid conflicts after the join
filtered_groups_renamed = filtered_groups.withColumnRenamed("max_count", "filtered_max_count")

# Perform an inner join between tie_breaker_df and filtered_groups_renamed
joined_df = tie_breaker_df.join(filtered_groups_renamed, 
                                (tie_breaker_df["max_personid"] == filtered_groups_renamed["max_personid"]) &
                                (tie_breaker_df["max_count_labcode"] == filtered_groups_renamed["max_count_labcode"]) &
                                (tie_breaker_df["max_count"] == filtered_groups_renamed["filtered_max_count"]),
                                "inner") \
                          .select(tie_breaker_df["max_personid"], 
                                  tie_breaker_df["max_count_labcode"], 
                                  tie_breaker_df["max_count"], 
                                  tie_breaker_df["max_count_updated_Interpretation"], 
                                  tie_breaker_df["mx_servicedate"], 
                                  filtered_groups_renamed["distinct_count"])

final_result_df = joined_df

final_result_df.show(100,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-----------------+---------+--------------------------------+--------------+--------------+
|max_personid                        |max_count_labcode|max_count|max_count_updated_Interpretation|mx_servicedate|distinct_count|
+------------------------------------+-----------------+---------+--------------------------------+--------------+--------------+
|063623e6-8398-419f-9695-a79723a9f9ea|59156-0          |1        |Normal                          |2021-10-21    |2             |
|3363e7a7-87ca-4d59-931e-68e597fa31ce|5209-2           |16       |High                            |2021-12-21    |2             |
|5b45ac11-b204-4dbf-b9f7-878ee5f5bde4|11075-9          |1        |Normal                          |2021-01-18    |2             |
|7e28074d-0094-425f-bc51-66e734a1e185|2081-8           |1        |Normal                          |2021-06-24    |2             |
|9be06bcc-e70f-4660-83b8-debfdeb4e0ce|22594-6          |1        |Normal                  

<IPython.core.display.Javascript object>

In [7]:
print("interpretation_counts", interpretation_counts.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

interpretation_counts 39691756


<IPython.core.display.Javascript object>